# 02 — Memory hierarchy and bandwidth

**Question.** How much traffic can each level of the memory system sustain, what does a dependent access cost, and how do the Triad, read-only and non-temporal-store kernels differ?

GB/s is kernel traffic (bytes the kernel reads and writes), not controller traffic. Working-set sizes are per thread for the hierarchy sweep (private arrays, first-touched by their owner) and total for the full-socket kernels.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('02_memory_hierarchy_and_bandwidth')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say = ctx.table, ctx.figure, ctx.say

## 1. Read bandwidth against working-set size

**Conditions.** One core and all 56 cores of socket 0, AVX-512 loads from private per-thread arrays, local memory. Hierarchy transitions appear as steps: L1 (48 KiB/core), L2 (2 MiB/core) and the shared L3 (112.5 MiB/socket). The one-core curve is the per-core capability at each level; the 56-core curve is the socket aggregate.

In [ ]:
h = report.hierarchy(ext)
one = h[h.notes == 'hier_read_1']
fig = plotting.line_by_config(one, 'per_thread_bytes', 'gbs', 'config', 'Read bandwidth · one core',
                              'Working set per core (bytes)', 'GB/s', log_y=True)
for edge, name in [(48 * 2**10, 'L1d 48 KiB'), (2 * 2**20, 'L2 2 MiB'), (112.5 * 2**20, 'L3 112.5 MiB')]:
    fig.add_vline(x=edge, line_dash='dot', line_color='#9AA3AF', annotation_text=name, annotation_position='top')
figure(fig, 'hierarchy_read_1core')

In [ ]:
full = h[h.notes == 'hier_read_56']
fig = plotting.line_by_config(full, 'per_thread_bytes', 'gbs', 'config', 'Read bandwidth · 56 cores, one socket',
                              'Working set per core (bytes)', 'GB/s', log_y=True)
fig.add_vline(x=112.5 * 2**20 / 56, line_dash='dot', line_color='#9AA3AF', annotation_text='L3 share per core', annotation_position='top')
figure(fig, 'hierarchy_read_56cores')

In [ ]:
levels, latency = report.level_summary(h)
table(levels.pivot(index='level', columns='config', values='gbs').reset_index())

In [ ]:
lv = levels.pivot(index='level', columns='config', values='gbs')
say(f"**Result.** Per core, L1 sustains {lv.loc['L1 (32 KiB)','flat_hbm']:.0f} GB/s, L2 {lv.loc['L2 (1 MiB)','flat_hbm']:.0f} GB/s and the L3 "
    f"{lv.loc['LLC (16 MiB)','flat_hbm']:.0f} GB/s; a single core reaches only {lv.loc['memory, 1 core (4 GiB)','flat_hbm']:.0f} GB/s from memory. "
    f"All 56 cores aggregate to {lv.loc['L1, 56 cores (32 KiB/core)','flat_hbm']/1000:.1f} TB/s in L1 and {lv.loc['L2, 56 cores (1 MiB/core)','flat_hbm']/1000:.1f} TB/s in L2, "
    f"against {lv.loc['memory, 56 cores (256 MiB/core)','flat_hbm']:.0f} GB/s (HBM) and {lv.loc['memory, 56 cores (256 MiB/core)','flat_ddr']:.0f} GB/s (DDR) from memory.")

**Implication.** Memory bandwidth grows with core count, not with a single core's capability: a thread needs data to fit in L2 to run at 100+ GB/s, while a thread streaming from memory is capped near 17–25 GB/s regardless of tier. The 16 KiB point reads lower than 32 KiB because a pass over 16 KiB takes only tens of nanoseconds, comparable to the clock read that ends each pass; it is a measurement overhead, not an L1 property.

## 2. Idle dependent-load latency

**Conditions.** One core chasing a random cycle of 64-byte nodes (seed 7); transparent huge pages are enabled system-wide, so translation cost is small. Results describe idle latency, not loaded latency.

In [ ]:
lat = h[h.notes == 'hier_latency']
fig = plotting.line_by_config(lat, 'per_thread_bytes', 'latency_ns', 'config', 'Dependent-load latency · one core',
                              'Working set (bytes)', 'ns per load', log_y=True)
for edge, name in [(48 * 2**10, 'L1d'), (2 * 2**20, 'L2'), (112.5 * 2**20, 'L3')]:
    fig.add_vline(x=edge, line_dash='dot', line_color='#9AA3AF', annotation_text=name, annotation_position='top')
figure(fig, 'hierarchy_latency')

In [ ]:
table(latency.pivot(index='level', columns='config', values='latency_ns').reset_index())

**Implication.** Cache levels are identical across configurations; only beyond the L3 do tiers differ. HBM's idle latency is higher than DDR's, so latency-bound code does not benefit from HBM; its benefit is bandwidth under many outstanding requests. In Cache mode the memory-side cache is a further HBM-latency step in front of DDR.

## 3. Triad, read-only and non-temporal Triad

**Conditions.** Socket 0, 56 cores, 8 GiB total, local memory (Flat), Cache mode at 8 GiB (HBM-resident). Triad moves 24 B per element as counted; if the hardware also read the destination line (write-allocate) the bus carries 32 B, so a Triad that is not write-allocate-free would read lower than the non-temporal variant, which never reads the destination.

In [ ]:
bw = report.bandwidth_summary(ext, thesis)
local = bw[(bw.locality == 'local') & (bw.socket_scope == 'single') & (bw.working_set_gib == 8)]
fig = plotting.go.Figure()
for cfg in ['flat_ddr', 'flat_hbm', 'cache']:
    part = local[local.config == cfg].groupby('kernel').gbs.mean()
    fig.add_bar(x=list(part.index), y=list(part.values), name=plotting.CONFIG_LABELS[cfg], marker_color=plotting.CONFIG_COLORS[cfg])
plotting.style(fig, 'Sustained bandwidth by kernel · one socket, 56 cores, 8 GiB', 'Kernel', 'GB/s', 480)
fig.update_layout(barmode='group')
figure(fig, 'kernel_comparison')
table(local.pivot_table(index='config', columns='kernel', values='gbs').reset_index())

In [ ]:
p = local.pivot_table(index='config', columns='kernel', values='gbs')
say(f"**Result.** Triad and non-temporal Triad agree to {100*abs(p.loc['flat_ddr','triad']/p.loc['flat_ddr','triad_nt']-1):.1f}% on DDR ({p.loc['flat_ddr','triad']:.0f} and {p.loc['flat_ddr','triad_nt']:.0f} GB/s) "
    f"and {100*abs(p.loc['flat_hbm','triad']/p.loc['flat_hbm','triad_nt']-1):.1f}% on HBM ({p.loc['flat_hbm','triad']:.0f} and {p.loc['flat_hbm','triad_nt']:.0f} GB/s), "
    f"so the destination line is not being read before it is written. Read-only is {p.loc['flat_ddr','read']/p.loc['flat_ddr','triad']-1:+.0%} on DDR ({p.loc['flat_ddr','read']:.0f} GB/s, the firmware-advertised 262) "
    f"and {p.loc['flat_hbm','read']/p.loc['flat_hbm','triad']-1:+.0%} on HBM ({p.loc['flat_hbm','read']:.0f} GB/s): reading is not faster than a mixed read/write stream on HBM.")

**Implication.** Full-line vector stores do not appear to cost an extra read here (consistent with the core skipping the read for whole-line writes), so counted Triad bandwidth is a fair proxy for bus traffic; this is inferred from equal rates, not observed with controller counters. For a roofline, the primary ceiling is the best sustained of the three kernels (ERT convention), with read-only shown as the decode-relevant slope.

## 4. Core count needed to saturate memory

In [ ]:
rs = report.read_scaling(ext)
rs0 = rs[rs.socket == 0]
fig = plotting.line_by_config(rs0, 'threads', 'gbs', 'config', 'Read bandwidth against core count · socket 0, 8 GiB',
                              'Cores', 'GB/s', error='std')
figure(fig, 'read_core_scaling')
sat = (rs0.groupby('config').apply(lambda g: g[g.gbs >= 0.9 * g.gbs.max()].threads.min(), include_groups=False)).rename('cores for 90% of peak').reset_index()
table(sat)

**Implication.** DDR saturates with roughly 28 cores while HBM keeps scaling to all 56: a workload that uses fewer cores than that (for example a server pinned to 55 cores competing with other work) leaves HBM bandwidth unused.

## 5. Cache mode against working-set size

**Conditions.** One socket, 56 cores, read-only and non-temporal Triad, local DDR-backed allocation of 8–128 GiB. The 64 GiB HBM capacity is a reference line, not a fitted transition.

In [ ]:
cap = bw[(bw.config == 'cache') & (bw.socket_scope == 'single') & (bw.socket == 0)]
fig = plotting.go.Figure()
for kernel, part in cap.groupby('kernel'):
    part = part.sort_values('working_set_gib')
    fig.add_scatter(x=part.working_set_gib, y=part.gbs, mode='lines+markers', name=kernel)
t = thesis[(thesis.memory_mode == 'cache') & (thesis.benchmark == 'stream_triad') & (thesis.notes == 'capacity') & (thesis.socket == 0)]
tt = t.groupby('working_set_gib').bandwidth_gbs.median()
fig.add_scatter(x=tt.index, y=tt.values, mode='lines+markers', name='triad (October)', line=dict(dash='dot'))
fig.add_vline(x=64, line_dash='dot', line_color='#9AA3AF', annotation_text='HBM 64 GiB', annotation_position='top')
plotting.style(fig, 'Cache-mode bandwidth against working set · socket 0', 'Working set (GiB)', 'GB/s', 500)
figure(fig, 'cache_capacity')
table(cap.pivot_table(index='working_set_gib', columns='kernel', values='gbs').reset_index())

In [ ]:
pv = cap.pivot_table(index='working_set_gib', columns='kernel', values='gbs')
say(f"**Result.** Read-only bandwidth falls from {pv.loc[8,'read']:.0f} GB/s at 8 GiB to {pv.loc[96,'read']:.0f} GB/s at 96 GiB and {pv.loc[128,'read']:.0f} GB/s at 128 GiB, "
    f"{pv.loc[128,'read']/pv.loc[8,'read']:.0%} of the HBM-resident rate; at 96 GiB it is {pv.loc[96,'read']/ctx.thesis[(ctx.thesis.memory_mode=='flat')&(ctx.thesis.memory_target=='ddr')&(ctx.thesis.benchmark=='stream_triad')&(ctx.thesis.notes=='core_scaling')&(ctx.thesis.threads==56)&(ctx.thesis.socket==0)].bandwidth_gbs.median():.0%} of Flat DDR's Triad.")

**Implication.** Cache-mode bandwidth must be quoted with its working-set size. Once the active data exceeds the 64 GiB memory-side cache, bandwidth drops *below* what Flat-mode DDR delivers for the same kernel, because every miss pays for the cache lookup and fill. A model whose weights do not fit in 64 GiB should not expect HBM-class bandwidth in Cache mode.

## 6. Can DDR and HBM bandwidth be combined? (October measurement)

In [ ]:
from hwbench.analysis import concurrent_table
table(concurrent_table(thesis).drop(columns=['cpu_bind']))

Combined DDR+HBM streams from disjoint core sets do not exceed the all-HBM rate for the same cores; component rates show interference relative to isolated controls. This measures two independent streams, not an application that mixes tiers.